# AI Call Moderator v3 — vLLM Serving, Guided JSON, Batched Turns, Staged Token Accounting

Continuation of v2, with the inference layer rebuilt on **vLLM** (pattern adapted from the
AMD `build_airbnb_agent_mcp` workshop). What this buys us:

| | v2 (transformers in-process) | v3 (vLLM server) |
|---|---|---|
| Output format | regex-extract JSON, can fail | **guided JSON decoding** — the server constrains generation to a JSON schema, malformed output is *impossible* |
| Throughput | one `generate()` at a time | **continuous batching** — all turns of a recorded call are analyzed concurrently |
| Token metering | counted manually with the tokenizer | exact `usage.prompt_tokens` / `usage.completion_tokens` per response, **accounted per stage and per call** |
| Model lifecycle | reloaded per notebook restart | served once, reused across restarts |

**Borrowed from the workshop:** vLLM launched in a Jupyter terminal with `--api-key` and
`--served-model-name`, the OpenAI-compatible client, and MCP for external data (Kaggle, as in v2).

**Deliberately NOT borrowed:** the PydanticAI agent + tool-calling loop. A compliance
moderator is a *deterministic pipeline*, not an open-ended agent — an agent loop would spend
extra round-trips and tool-choice tokens deciding what we already know must happen on every
turn. Accuracy and token efficiency both favor direct, schema-constrained calls.

```
Kaggle MCP (OAuth 2.0) ──> recordings ──> Whisper ──> speaker separation + role ID
                                                            │
                                          all turns of a call, CONCURRENTLY
                                                            ▼
                              vLLM server (guided JSON, continuous batching)
                                                            │
                                                            ▼
                       deterministic escalation controller ──> end-of-call report
                                                            │
                                                            ▼
                       token accounting: per stage, per call, combined
```

## 1. Launch the vLLM server (one-time, in a terminal)

Open a **terminal tab** in Jupyter (`+` → Terminal) and run:

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-4B-Instruct-2507 \
    --served-model-name call-moderator-llm \
    --api-key local-key-123 \
    --port 8000 \
    --max-model-len 16384 \
    --gpu-memory-utilization 0.85
```

Notes:

- `Qwen3-4B-Instruct-2507` (not the thinking variant the workshop used) — no `<think>` token
  overhead; every completion token is answer, which matters when tokens are graded.
- `--max-model-len 16384` covers our longest prompt (a full transcript for the end-of-call
  report) while reducing KV-cache reservation; raise it for very long calls.
- `--gpu-memory-utilization 0.85` leaves headroom for Whisper on the same GPU.
- No `--enable-auto-tool-choice` / `--tool-call-parser` — we make direct schema-constrained
  calls, not agent tool calls.
- Optionally monitor the GPU in a second terminal with `watch rocm-smi`.

When the server logs `Application startup complete`, run the cells below.

In [ ]:
# 2. Environment setup — installs only what's missing (vLLM runs in its own process)
import importlib, subprocess, sys

def ensure(*packages):
    for package in packages:
        module_name = package.replace("-", "_")
        try:
            importlib.import_module(module_name)
        except ImportError:
            print(f"installing {package} ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

ensure("openai", "mcp", "httpx", "torch", "transformers", "librosa", "soundfile")

import os, json, re, time, asyncio
from openai import OpenAI, AsyncOpenAI

BASE_URL          = "http://localhost:8000/v1"
SERVED_MODEL_NAME = "call-moderator-llm"
API_KEY           = "local-key-123"

sync_client  = OpenAI(base_url=BASE_URL, api_key=API_KEY)
async_client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)

# health check — fails loudly if the vLLM server isn't up yet
available_models = [m.id for m in sync_client.models.list().data]
print("vLLM is serving:", available_models)
assert SERVED_MODEL_NAME in available_models, "served-model-name mismatch — check the serve command"

## 2. Stage-tagged generation with guided JSON

Every LLM call goes through one function, `generate_json`, which does three things:

1. **Guided decoding** — passes a JSON schema via `extra_body={"guided_json": ...}`; vLLM
   constrains token sampling to the schema's grammar, so output is *always* valid JSON that
   matches the schema. No regex extraction, no retries, no wasted tokens on malformed output.
2. **Stage accounting** — every call is tagged with a pipeline stage
   (`speaker_split`, `role_identification`, `turn_moderation`, `end_of_call_report`) and the
   server's exact `usage` counts are accumulated per stage AND per moderated call.
3. **Determinism** — `temperature=0` (greedy), reproducible runs.

In [ ]:
from collections import defaultdict

STAGE_TOKEN_USAGE    = defaultdict(lambda: {"calls": 0, "prompt": 0, "completion": 0})
PER_CALL_TOKEN_USAGE = defaultdict(lambda: {"calls": 0, "prompt": 0, "completion": 0})

def record_usage(stage: str, usage, call_id: str = None):
    buckets = [STAGE_TOKEN_USAGE[stage]]
    if call_id is not None:
        buckets.append(PER_CALL_TOKEN_USAGE[call_id])
    for bucket in buckets:
        bucket["calls"]      += 1
        bucket["prompt"]     += usage.prompt_tokens
        bucket["completion"] += usage.completion_tokens

async def generate_json(stage: str, system_prompt: str, user_prompt: str,
                        json_schema: dict, max_tokens: int = 96, call_id: str = None) -> dict:
    """One schema-constrained, greedy LLM call. Usage is recorded per stage and per call."""
    response = await async_client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user",   "content": user_prompt}],
        temperature=0,
        max_tokens=max_tokens,
        extra_body={"guided_json": json_schema},   # vLLM structured output
    )
    record_usage(stage, response.usage, call_id)
    try:
        return json.loads(response.choices[0].message.content)
    except json.JSONDecodeError:
        return {}

# smoke test: guided decoding + usage metering (Jupyter supports top-level await)
smoke_schema = {"type": "object", "properties": {"sentiment": {"enum": [-2, -1, 0, 1, 2]}},
                "required": ["sentiment"]}
smoke_result = await generate_json("smoke_test", "Rate customer sentiment.",
                                   'Customer said: "this is great, thanks!" JSON:',
                                   smoke_schema, max_tokens=16)
print("guided JSON result :", smoke_result)
print("smoke-test usage   :", dict(STAGE_TOKEN_USAGE["smoke_test"]))

## 3. Policy, standard procedure & zero-token pre-screener

Unchanged from v2 (the enforcement content is the product; the serving layer is what v3
upgrades). Violation codes keep prompts and outputs tiny.

In [ ]:
# Violation catalog: code -> (severity, definition)
POLICY = {
    # CRITICAL - escalate immediately
    "C1": ("critical", "Improper sensitive data: asking for or reading out a full SSN, "
                       "full card number, CVV, or account password"),
    "C2": ("critical", "Threats, harassment, hate speech, or targeted abuse at a person"),
    "C3": ("critical", "Unethical conduct: soliciting tips/bribes for favors, off-the-books "
                       "deals, falsifying records, or advising someone to lie"),
    # HIGH
    "C4": ("high",     "Unauthorized commitment: promising refunds over $50, compensation, "
                       "or outcomes outside stated policy"),
    "R1": ("high",     "Rep makes/offers account changes WITHOUT identity verification"),
    "R3": ("high",     "Disclosing internal-only data or another customer's information"),
    # MEDIUM
    "R2": ("medium",   "Rep rudeness: dismissive, sarcastic, blaming, or hostile tone"),
}
SEVERITY_BY_CODE = {code: severity for code, (severity, _) in POLICY.items()}

STANDARD_PROCEDURE = {
    "S1": "Greeting: rep gives name + company and offers help",
    "S2": "Identity verification before account discussion/changes",
    "S3": "Resolution recap: rep restates what was done / next steps",
    "S4": "Closing: rep asks if anything else is needed and closes politely",
}

COMPANY_POLICY_ALLOWANCES = (
    "Refunds up to $50 may be issued instantly; larger amounts require supervisor approval. "
    "Identity verification = full name + last 4 of account + billing ZIP (NEVER full SSN, "
    "full card number, CVV, or password). Reps may offer one goodwill credit up to $25 per call."
)

PRESCREEN_PATTERNS = {
    "C1": [r"\b(full|entire|whole|complete)\b.{0,40}\b(ssn|social security|card number)",
           r"\bcvv\b", r"\bsecurity code\b.{0,20}\bback\b", r"\byour password\b"],
    "C2": [r"\b(idiot|stupid|moron|pathetic|shut up)\b",
           r"\bi('ll| will) (sue|hurt|ruin|find) you\b"],
    "C3": [r"\b(venmo|cash app|tip me|kickback)\b", r"\boff[- ]the[- ]books\b"],
    "C4": [r"(refund|credit|comp)\w*\b.{0,40}\$\s?\d{3,}",
           r"\$\s?\d{3,}.{0,40}\b(refund|credit)", r"\bi (promise|guarantee)\b"],
}

def keyword_prescreen(text: str) -> list:
    lowered = text.lower()
    return [code for code, patterns in PRESCREEN_PATTERNS.items()
            if any(re.search(pattern, lowered) for pattern in patterns)]

print(f"{len(POLICY)} violation codes, {len(STANDARD_PROCEDURE)} standard-procedure items loaded")

## 4. Moderation engine — schemas, prompts, and concurrent turn analysis

**Why turns can run concurrently:** each turn's analysis needs only the *transcript* of the
preceding turns — which, for a recorded call, is fully known upfront. The per-turn LLM
judgments do not feed each other; only the *escalation controller* is sequential, and it runs
after the analyses as cheap deterministic code that finds the earliest rule trigger.
So for offline/QA moderation we `asyncio.gather` every turn of a call and let vLLM's
continuous batching process them together. (Live streaming would simply revert to one call
per incoming turn — same code path, no batching.)

Guided JSON schemas pin every output: sentiment must be one of −2…2, violation codes must be
real codes, the rating must be 1–5. The model *cannot* answer off-schema.

In [ ]:
TURN_ANALYSIS_SCHEMA = {
    "type": "object",
    "properties": {
        "sentiment":  {"enum": [-2, -1, 0, 1, 2]},
        "violations": {"type": "array", "items": {"enum": list(POLICY)}},
        "reason":     {"type": "string", "maxLength": 90},
    },
    "required": ["sentiment", "violations", "reason"],
}

REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "rating":    {"enum": [1, 2, 3, 4, 5]},
        "procedure": {"type": "object",
                      "properties": {item: {"enum": [0, 1]} for item in STANDARD_PROCEDURE},
                      "required": list(STANDARD_PROCEDURE)},
        "summary":   {"type": "string", "maxLength": 200},
        "coaching":  {"type": "string", "maxLength": 140},
    },
    "required": ["rating", "procedure", "summary", "coaching"],
}

_policy_lines = "\n".join(f"{code}: {definition} [{severity}]"
                          for code, (severity, definition) in POLICY.items())

MODERATOR_SYSTEM_PROMPT = (
    "You are a strict call-center compliance moderator. Judge ONLY the LATEST turn, using context.\n"
    "Violation codes:\n" + _policy_lines + "\n"
    "Company policy: " + COMPANY_POLICY_ALLOWANCES + "\n"
    "Important: a frustrated/venting customer is NOT a violation unless it becomes threats or "
    "targeted abuse (C2). A rep correctly REFUSING an improper request is NOT a violation. "
    "Mentioning a policy limit is NOT a promise. "
    "Report sentiment as the customer's sentiment evident right now (-2..2)."
)

REPORT_SYSTEM_PROMPT = (
    "You are a call-center QA analyst. Given a transcript, output: the predicted 1-5 customer "
    "satisfaction rating, a 0/1 mark for each standard-procedure item, a <=25 word summary, "
    "and <=18 words of coaching advice for the rep.\n"
    "Standard-procedure items: " + json.dumps(STANDARD_PROCEDURE)
)

async def analyze_turn(previous_turns: list, speaker_role: str, turn_text: str, call_id: str) -> dict:
    context = "\n".join(f"{role}: {text}" for role, text in previous_turns[-4:]) or "(call start)"
    keyword_hints = keyword_prescreen(turn_text)
    hint_note = f"\nRegex hints to verify (may be false alarms): {keyword_hints}" if keyword_hints else ""
    user_prompt = (f'Context:\n{context}\n\nLATEST {speaker_role.upper()} turn: '
                   f'"{turn_text}"{hint_note}')
    return await generate_json("turn_moderation", MODERATOR_SYSTEM_PROMPT, user_prompt,
                               TURN_ANALYSIS_SCHEMA, max_tokens=72, call_id=call_id)

MAX_TURNS_PER_CALL = 24

async def moderate_call(call_id: str, turns: list) -> dict:
    """Analyze every turn concurrently (vLLM batches them), then apply escalation rules in order."""
    turns = turns[:MAX_TURNS_PER_CALL]
    analysis_started = time.time()
    analyses = await asyncio.gather(*[
        analyze_turn(turns[:index], speaker_role, turn_text, call_id)
        for index, (speaker_role, turn_text) in enumerate(turns)
    ])
    analysis_seconds = time.time() - analysis_started

    # deterministic escalation controller — sequential scan finds the EARLIEST trigger
    violations_log, customer_sentiment_history = [], []
    escalated, escalation_reason, escalation_turn = False, None, None
    print("=" * 78); print(f"CALL {call_id}"); print("=" * 78)
    for turn_number, ((speaker_role, turn_text), analysis) in enumerate(zip(turns, analyses), start=1):
        violations = [code for code in analysis.get("violations", []) if code in POLICY]
        sentiment = analysis.get("sentiment", 0)
        violations_log += [(turn_number, speaker_role, code) for code in violations]
        if speaker_role == "customer":
            customer_sentiment_history.append(sentiment)
        if not escalated:
            severities = [SEVERITY_BY_CODE[code] for _, _, code in violations_log]
            recent_sentiment = customer_sentiment_history[-2:]
            if "critical" in severities:
                escalated, escalation_reason = True, f"critical violation {violations} (rule 1)"
            elif severities.count("high") >= 2:
                escalated, escalation_reason = True, "two high-severity violations (rule 2)"
            elif len(recent_sentiment) == 2 and all(s <= -2 for s in recent_sentiment):
                escalated, escalation_reason = True, "sustained very negative customer sentiment (rule 3)"
            if escalated:
                escalation_turn = turn_number
        sentiment_note = f" (sentiment {sentiment:+d})" if speaker_role == "customer" else ""
        violation_note = f"   <- VIOLATION {violations}: {analysis.get('reason', '')}" if violations else ""
        print(f"[{turn_number:02d}] {speaker_role:>8}{sentiment_note}: {turn_text[:160]}{violation_note}")
        if escalation_turn == turn_number:
            print(f"     *** ESCALATED at turn {turn_number} -> {escalation_reason} ***")

    transcript = "\n".join(f"{role}: {text}" for role, text in turns)
    report = await generate_json("end_of_call_report", REPORT_SYSTEM_PROMPT,
                                 f"Transcript:\n{transcript}", REPORT_SCHEMA,
                                 max_tokens=160, call_id=call_id)
    procedure_marks = ", ".join(f"{item}={mark}" for item, mark in report.get("procedure", {}).items())
    print(f"\n  end-of-call | rating: {report.get('rating')}/5 | standard procedure: {procedure_marks}")
    print(f"  summary : {report.get('summary', '')}")
    print(f"  coaching: {report.get('coaching', '')}")
    print(f"  speed   : {len(turns)} turn analyses in {analysis_seconds:.1f}s "
          f"({len(turns)/max(analysis_seconds, 0.001):.1f} turns/s, batched)\n")
    return {"escalated": escalated, "escalation_reason": escalation_reason,
            "escalation_turn": escalation_turn,
            "violations": sorted({code for _, _, code in violations_log}),
            "report": report, "analysis_seconds": analysis_seconds, "turn_count": len(turns)}

print("moderation engine ready")

## 5. Pull recordings from Kaggle via MCP (OAuth 2.0)

Identical to v2: the notebook is an MCP client to Kaggle's official server with OAuth 2.0
(manual paste flow for the headless lab), and tool names are discovered at runtime.

In [ ]:
from urllib.parse import urlparse, parse_qs
from mcp.client.auth import OAuthClientProvider, TokenStorage
from mcp.shared.auth import OAuthClientMetadata

KAGGLE_MCP_URL = "https://www.kaggle.com/mcp"

class InMemoryTokenStorage(TokenStorage):
    """Keeps OAuth tokens in RAM for this session only (no files written)."""
    def __init__(self):
        self.tokens, self.client_info = None, None
    async def get_tokens(self):            return self.tokens
    async def set_tokens(self, tokens):    self.tokens = tokens
    async def get_client_info(self):       return self.client_info
    async def set_client_info(self, info): self.client_info = info

async def show_authorization_url(authorization_url: str):
    print("1) Open this URL in your browser and approve access:\n")
    print(authorization_url)

async def wait_for_pasted_callback():
    pasted_url = input("\n2) After approving, your browser lands on a localhost URL that fails "
                       "to load (expected). Paste that FULL URL here: ").strip()
    query = parse_qs(urlparse(pasted_url).query)
    return query["code"][0], query.get("state", [None])[0]

token_storage = InMemoryTokenStorage()

kaggle_oauth = OAuthClientProvider(
    server_url=KAGGLE_MCP_URL,
    client_metadata=OAuthClientMetadata(
        client_name="call-moderator-v3-notebook",
        redirect_uris=["http://localhost:8765/callback"],
        grant_types=["authorization_code", "refresh_token"],
        response_types=["code"],
    ),
    storage=token_storage,
    redirect_handler=show_authorization_url,
    callback_handler=wait_for_pasted_callback,
)

from contextlib import AsyncExitStack
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

connection_stack = AsyncExitStack()
read_stream, write_stream, _ = await connection_stack.enter_async_context(
    streamablehttp_client(KAGGLE_MCP_URL, auth=kaggle_oauth)
)
kaggle_session = await connection_stack.enter_async_context(ClientSession(read_stream, write_stream))
await kaggle_session.initialize()

KAGGLE_TOOLS = (await kaggle_session.list_tools()).tools
print(f"Connected — {len(KAGGLE_TOOLS)} Kaggle tools available:\n")
for tool in KAGGLE_TOOLS:
    first_description_line = (tool.description or "").splitlines()[0][:70] if tool.description else ""
    print(f"  {tool.name:<42} {first_description_line}")

In [ ]:
import httpx, zipfile, pathlib

def result_payload(call_result):
    """Flatten an MCP CallToolResult into Python data (JSON-decoded when possible)."""
    text = "\n".join(c.text for c in call_result.content if getattr(c, "text", None))
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text

async def call_kaggle(tool_name: str, argument_variants: list):
    """Call a Kaggle tool, trying several plausible argument spellings in order."""
    last_error = None
    for arguments in argument_variants:
        result = await kaggle_session.call_tool(tool_name, arguments)
        if not result.isError:
            return result_payload(result)
        last_error = result_payload(result)
    raise RuntimeError(f"{tool_name} failed with all argument variants. Last error: {last_error}")

def find_tool(*name_keywords):
    for tool in KAGGLE_TOOLS:
        if all(keyword in tool.name.lower() for keyword in name_keywords):
            return tool.name
    return None

search_tool_name   = find_tool("search", "dataset") or find_tool("dataset", "list")
download_tool_name = find_tool("download", "dataset") or find_tool("dataset", "url")

SEARCH_QUERY = "call center audio english"
search_results = await call_kaggle(search_tool_name, [
    {"query": SEARCH_QUERY}, {"search": SEARCH_QUERY}, {"q": SEARCH_QUERY},
])
preview = search_results if isinstance(search_results, str) else json.dumps(search_results, indent=2)
print(preview[:1500])

DATASET_REFERENCE = "unidpro/call-center-audio"   # owner/slug — change based on the search output
owner_slug, dataset_slug = DATASET_REFERENCE.split("/")

download_info = await call_kaggle(download_tool_name, [
    {"dataset": DATASET_REFERENCE},
    {"ref": DATASET_REFERENCE},
    {"dataset_ref": DATASET_REFERENCE},
    {"owner_slug": owner_slug, "dataset_slug": dataset_slug},
])

def first_url(payload):
    text = payload if isinstance(payload, str) else json.dumps(payload)
    match = re.search(r"https?://[^\s\"']+", text)
    return match.group(0) if match else None

download_url = first_url(download_info)
DATA_DIR = pathlib.Path("kaggle_call_data")
DATA_DIR.mkdir(exist_ok=True)
archive_path = DATA_DIR / "dataset_download"

request_headers = {}
stored_tokens = await token_storage.get_tokens()
if stored_tokens:
    request_headers["Authorization"] = f"Bearer {stored_tokens.access_token}"

with httpx.stream("GET", download_url, headers=request_headers,
                  follow_redirects=True, timeout=300) as response:
    response.raise_for_status()
    with open(archive_path, "wb") as output_file:
        for chunk in response.iter_bytes():
            output_file.write(chunk)
print(f"downloaded {archive_path.stat().st_size / 1e6:.1f} MB")

if zipfile.is_zipfile(archive_path):
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DATA_DIR)

AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}
audio_files = sorted(p for p in DATA_DIR.rglob("*") if p.suffix.lower() in AUDIO_EXTENSIONS)
transcript_files = sorted(p for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".csv", ".txt"})
print(f"{len(audio_files)} audio files, {len(transcript_files)} transcript files")

## 6. Transcription, speaker separation & role identification

Same logic as v2, now with guided JSON: stereo files separate physically by channel (free);
mono files get one schema-constrained LLM call (`speaker_split` stage). Roles are decided by
keyword marker scores first (0 tokens), with a tiny schema-constrained tiebreak
(`role_identification` stage) only when ambiguous.

In [ ]:
import torch, numpy as np, librosa
from transformers import pipeline

speech_recognizer = pipeline("automatic-speech-recognition", model="openai/whisper-small",
                             device=0 if torch.cuda.is_available() else -1,
                             chunk_length_s=30, return_timestamps=True)

def transcribe_samples(samples, sampling_rate):
    output = speech_recognizer({"raw": samples, "sampling_rate": sampling_rate})
    return [{"start": (chunk["timestamp"][0] or 0.0), "text": chunk["text"].strip()}
            for chunk in output["chunks"] if chunk["text"].strip()]

SPLIT_SCHEMA = {
    "type": "object",
    "properties": {"turns": {"type": "array", "items": {
        "type": "object",
        "properties": {"role": {"enum": ["rep", "customer"]}, "text": {"type": "string"}},
        "required": ["role", "text"]}}},
    "required": ["turns"],
}
TRANSCRIPT_SPLIT_PROMPT = (
    "You segment a raw call-center transcript into speaker turns and label each turn "
    "'rep' (the service representative) or 'customer'. Keep the original wording; do not invent text."
)

async def split_mono_transcript(full_text: str, call_id: str, max_characters: int = 3500) -> list:
    response = await generate_json("speaker_split", TRANSCRIPT_SPLIT_PROMPT,
                                   f"Transcript:\n{full_text[:max_characters]}",
                                   SPLIT_SCHEMA, max_tokens=1024, call_id=call_id)
    return [{"start": index, "speaker": turn["role"], "text": turn["text"]}
            for index, turn in enumerate(response.get("turns", [])) if turn.get("text")]

async def utterances_from_audio(audio_path, call_id):
    samples, sampling_rate = librosa.load(audio_path, sr=16000, mono=False)
    if samples.ndim == 2 and samples.shape[0] == 2:        # stereo: one party per channel
        utterances = []
        for channel_index in (0, 1):
            for utterance in transcribe_samples(samples[channel_index], sampling_rate):
                utterances.append({**utterance, "speaker": f"speaker_{channel_index}"})
        return sorted(utterances, key=lambda u: u["start"]), "stereo"
    mono_samples = samples if samples.ndim == 1 else samples.mean(axis=0)
    full_text = " ".join(u["text"] for u in transcribe_samples(mono_samples, sampling_rate))
    return await split_mono_transcript(full_text, call_id), "mono"

REP_MARKER_PHRASES = (
    "thank you for calling", "how can i help", "how may i help", "this is",
    "let me check", "let me look", "i can help", "i apologize", "i'm sorry for",
    "is there anything else", "have a great day", "your account shows",
)
CUSTOMER_MARKER_PHRASES = (
    "my bill", "my account", "i was charged", "i'm calling about", "im calling about",
    "i want a refund", "i need help", "cancel my", "not working", "speak to a",
    "my order", "my service",
)
ROLE_SCHEMA = {"type": "object", "properties": {"rep": {"enum": ["speaker_0", "speaker_1"]}},
               "required": ["rep"]}
ROLE_TIEBREAK_PROMPT = ("Two speakers from a call-center call. "
                        "Decide which one is the customer-service rep.")

async def identify_roles(utterances: list, call_id: str) -> dict:
    """Keyword marker scores first (0 tokens); schema-constrained LLM tiebreak only on ties."""
    marker_scores = {}
    for utterance in utterances:
        text = utterance["text"].lower()
        score_change = (sum(phrase in text for phrase in REP_MARKER_PHRASES)
                        - sum(phrase in text for phrase in CUSTOMER_MARKER_PHRASES))
        marker_scores[utterance["speaker"]] = marker_scores.get(utterance["speaker"], 0) + score_change
    speakers = sorted(marker_scores)
    if len(speakers) == 1:
        return {speakers[0]: "rep"}
    score_margin = abs(marker_scores[speakers[0]] - marker_scores[speakers[1]])
    if score_margin >= 2:
        rep_speaker = max(speakers, key=lambda s: marker_scores[s])
    else:
        call_opening = "\n".join(f"{u['speaker']}: {u['text']}" for u in utterances[:8])
        answer = await generate_json("role_identification", ROLE_TIEBREAK_PROMPT,
                                     f"Call opening:\n{call_opening}",
                                     ROLE_SCHEMA, max_tokens=24, call_id=call_id)
        rep_speaker = answer.get("rep", speakers[0])
    return {speaker: ("rep" if speaker == rep_speaker else "customer") for speaker in speakers}

def merge_consecutive_turns(labeled_turns: list) -> list:
    merged = []
    for role, text in labeled_turns:
        if merged and merged[-1][0] == role:
            merged[-1] = (role, merged[-1][1] + " " + text)
        else:
            merged.append((role, text))
    return merged

print("transcription + speaker logic ready")

## 7. Run end-to-end

Download → transcribe → separate → identify roles → **batched** moderation → report.
Text fallback (CSV transcripts) is automatic. Multiple calls are also moderated concurrently —
vLLM batches across calls too.

In [ ]:
MAX_CALLS_TO_MODERATE = 3

async def prepare_turns(audio_path, call_id):
    utterances, separation_mode = await utterances_from_audio(audio_path, call_id)
    print(f"{call_id}: {len(utterances)} utterances ({separation_mode} separation)")
    if separation_mode == "stereo":
        speaker_roles = await identify_roles(utterances, call_id)
        print(f"  roles: {speaker_roles}")
        labeled_turns = [(speaker_roles[u["speaker"]], u["text"]) for u in utterances]
    else:
        labeled_turns = [(u["speaker"], u["text"]) for u in utterances]   # mono path is pre-labeled
    return merge_consecutive_turns(labeled_turns)

moderation_results = {}
pipeline_started = time.time()

if audio_files:
    selected_files = audio_files[:MAX_CALLS_TO_MODERATE]
    turns_per_call = {p.name: await prepare_turns(p, p.name) for p in selected_files}
    call_outcomes = await asyncio.gather(*[
        moderate_call(call_id, turns) for call_id, turns in turns_per_call.items()
    ])
    moderation_results = dict(zip(turns_per_call, call_outcomes))
elif transcript_files:
    ensure("pandas"); import pandas as pd
    table = pd.read_csv(transcript_files[0])
    text_column = max(table.columns, key=lambda c: table[c].astype(str).str.len().mean())
    print(f"text fallback: using column '{text_column}' of {transcript_files[0].name}")
    turns_per_call = {}
    for row_index, raw_transcript in enumerate(table[text_column].head(MAX_CALLS_TO_MODERATE)):
        call_id = f"transcript_{row_index}"
        utterances = await split_mono_transcript(str(raw_transcript), call_id)
        turns_per_call[call_id] = merge_consecutive_turns([(u["speaker"], u["text"]) for u in utterances])
    call_outcomes = await asyncio.gather(*[
        moderate_call(call_id, turns) for call_id, turns in turns_per_call.items()
    ])
    moderation_results = dict(zip(turns_per_call, call_outcomes))
else:
    print("No audio or transcript files found — check the download cell output.")

pipeline_seconds = time.time() - pipeline_started
print(f"\nmoderated {len(moderation_results)} call(s) in {pipeline_seconds:.1f}s total")

## 8. Token accounting — per stage, per call, combined

Every number below comes from vLLM's exact `usage` field, not estimates. The stage table
shows where the budget goes (turn moderation should dominate; speaker logic should be near
zero thanks to channel separation and the keyword-first role classifier).

In [ ]:
def print_usage_table(title, usage_by_key):
    print(title)
    print(f"  {'key':<28}{'calls':>7}{'prompt':>10}{'completion':>12}{'total':>10}{'avg/call':>10}")
    print("  " + "-" * 77)
    total = {"calls": 0, "prompt": 0, "completion": 0}
    for key, usage in sorted(usage_by_key.items()):
        row_total = usage["prompt"] + usage["completion"]
        average = row_total // max(usage["calls"], 1)
        print(f"  {key:<28}{usage['calls']:>7}{usage['prompt']:>10,}{usage['completion']:>12,}"
              f"{row_total:>10,}{average:>10,}")
        for field in total:
            total[field] += usage[field]
    grand_total = total["prompt"] + total["completion"]
    print("  " + "-" * 77)
    print(f"  {'TOTAL':<28}{total['calls']:>7}{total['prompt']:>10,}{total['completion']:>12,}"
          f"{grand_total:>10,}{grand_total // max(total['calls'], 1):>10,}")
    print()
    return grand_total

stage_total    = print_usage_table("TOKENS BY PIPELINE STAGE", STAGE_TOKEN_USAGE)
per_call_total = print_usage_table("TOKENS BY MODERATED CALL", PER_CALL_TOKEN_USAGE)

print(f"COMBINED total (all stages)        : {stage_total:,} tokens "
      f"across {sum(u['calls'] for u in STAGE_TOKEN_USAGE.values())} LLM calls")
for call_id, outcome in moderation_results.items():
    usage = PER_CALL_TOKEN_USAGE[call_id]
    tokens_per_turn = (usage["prompt"] + usage["completion"]) // max(outcome["turn_count"], 1)
    print(f"  {call_id:<30} escalated={outcome['escalated']}"
          f"{' (turn ' + str(outcome['escalation_turn']) + ')' if outcome['escalated'] else ''} "
          f"violations={outcome['violations'] or '-'} "
          f"rating={outcome['report'].get('rating')}/5 "
          f"~{tokens_per_turn:,} tokens/turn")

## Design notes

- **From the workshop notebook we kept:** vLLM served via terminal with `--api-key` /
  `--served-model-name`, the OpenAI-compatible client, MCP for external data. **We skipped**
  PydanticAI's agent loop: a moderator pipeline is deterministic, so direct schema-constrained
  calls are both more accurate (guided decoding cannot produce off-schema output) and cheaper
  (no tool-choice round-trips or agent scaffolding tokens).
- **Concurrency is correctness-preserving:** per-turn judgments depend only on the known
  transcript, never on each other; the escalation decision is recomputed sequentially by
  deterministic code, so batched results are identical to one-at-a-time results — just faster.
- **Live deployment:** the same `analyze_turn` is called once per incoming turn (no batching),
  and the escalation controller runs incrementally — identical logic, streaming latency.
- **Tuning:** `--max-model-len` bounds the report prompt; `MAX_TURNS_PER_CALL` and
  `MAX_CALLS_TO_MODERATE` bound spend; swap the served model for a bigger judge without
  touching notebook code (just the serve command).